# Sprint 3 — Webinar 12 (Práctico) — Student Version

## SQL aplicado a KPIs de negocio en VS Code, DuckDB y Parquet local

> **Duración sugerida:** 100 minutos  
> **Herramienta:** VS Code + SQLTools + DuckDB Driver  
> **Motor:** DuckDB local  
> **Dataset:** archivos Parquet locales en `data/`  
> **Formato:** práctica guiada usando archivos `.sql`.

Esta sesión usa el mismo proyecto local de la clase anterior:

```text
bookstore_finance_sql_vscode
```

La práctica se resuelve principalmente en:

```text
sql/07_w12_practica_guiada.sql
sql/08_w12_retos.sql
```

Este notebook sirve como guía de estudiante: contiene consignas, plantillas, espacios de interpretación y checklist final.

<div style="text-align: center">

## Objetivo central

Resolver ejercicios completos de KPIs de negocio con SQL, pasando de la pregunta a la consulta y de la consulta a una interpretación accionable.

</div>

## Agenda de práctica (100 minutos)

| Bloque | Tiempo | Actividad |
|---|---:|---|
| 0. Verificación técnica | 10 min | Conexión, vistas y sanity checks |
| 1. Método de trabajo | 5 min | Estrategia para resolver ejercicios SQL |
| 2. Ejercicio 0 | 10 min | Estado de orden y método de pago |
| 3. Ejercicio 1 | 15 min | Órdenes por segmento |
| 4. Ejercicio 2 | 15 min | Revenue por categoría |
| 5. Ejercicio 3 | 15 min | Top 5 productos |
| 6. Ejercicio 4 | 15 min | Margen bruto por categoría |
| 7. Ejercicio 5 | 10 min | Promo vs no promo |
| 8. Ejercicio 6 y cierre | 5 min | ROI por campaña y reflexión |

## Objetivos de aprendizaje

Al finalizar la práctica, podrás:

- Verificar que las vistas locales estén disponibles en DuckDB.
- Resolver consultas con `JOIN`, `GROUP BY`, `CASE`, `HAVING`, `ORDER BY` y `LIMIT`.
- Calcular revenue, ticket promedio, margen y ROI.
- Comparar grupos de negocio: segmentos, categorías, productos y campañas.
- Interpretar resultados en lenguaje de negocio.
- Trabajar en archivos `.sql`, preparando el terreno para Git y notebooks de Jupyter.

### Mi meta personal

**Meta:** ______________________________________________________________

---
# Bloque 0 · Verificación técnica

Antes de resolver:

1. Abre VS Code.
2. Abre la carpeta completa `bookstore_finance_sql_vscode`.
3. Conéctate a `bookstore.duckdb` desde SQLTools.
4. Ejecuta:

```text
sql/00_test_connection.sql
sql/01_setup_views_relative.sql
sql/02_data_dictionary_and_sanity_checks.sql
```

Si la ruta relativa falla, usa:

```text
sql/01b_setup_views_absolute_path_template.sql
```

### Checklist

- ☐ Veo `customers`.
- ☐ Veo `products`.
- ☐ Veo `orders`.
- ☐ Veo `order_items`.
- ☐ Veo `payments`.
- ☐ Veo `campaigns`.

## 0.1 Validación rápida

```sql
SELECT 'customers' AS tabla, COUNT(*) AS filas FROM customers
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'payments', COUNT(*) FROM payments
UNION ALL
SELECT 'campaigns', COUNT(*) FROM campaigns;
```

### Registro

| Tabla | Filas esperadas | Filas obtenidas | ¿Listo? |
|---|---:|---:|---|
| `customers` | 200 | | |
| `products` | 80 | | |
| `orders` | 600 | | |
| `order_items` | 1125 | | |
| `payments` | 600 | | |
| `campaigns` | 3 | | |

In [ ]:
-- Validación rápida
-- Copia aquí tu sanity check o registra notas.

---
# Bloque 1 · Método de trabajo para cada ejercicio

Antes de escribir SQL, responde:

1. ¿Cuál es la pregunta de negocio?
2. ¿Qué tabla contiene la métrica?
3. ¿Qué tabla contiene el contexto?
4. ¿Cuál es la llave del `JOIN`?
5. ¿Necesito filtrar estados?
6. ¿Debo agrupar?
7. ¿Cómo voy a interpretar el resultado?

Regla de negocio sugerida:

```sql
WHERE o.status NOT IN ('cancelled', 'refunded')
```

Usa esta regla cuando el ejercicio busque un KPI financiero operativo.

---
# Ejercicio 0 · Estado de orden y método de pago (Nivel: ⭐ Básico)

## Objetivo

Calcular, por `status` y `payment_method`:

- número de órdenes;
- total pagado.

## Antes de escribir

| Pregunta | Respuesta |
|---|---|
| Tabla de órdenes | |
| Tabla de pagos | |
| Llave de unión | |
| Métrica principal | |
| Agrupaciones necesarias | |

## Borrador

```sql
SELECT
    __________________,
    __________________,
    COUNT(DISTINCT __________________) AS n_orders,
    SUM(__________________) AS total_pagado
FROM __________________ AS o
JOIN __________________ AS p
    ON __________________
GROUP BY
    __________________,
    __________________
ORDER BY __________________ DESC;
```

## Interpretación

**Qué combinación concentra más valor pagado:** ____________________________

**Qué podría significar para el negocio:** _________________________________

In [ ]:
-- Ejercicio 0
-- Escribe tu consulta aquí.

---
# Ejercicio 1 · Órdenes por segmento de cliente (Nivel: ⭐ Básico)

## Objetivo

Obtener por `segment`:

- número de órdenes;
- total pagado;
- ticket promedio.

## Pistas

- `customers` se une con `orders` por `customer_id`.
- `orders` se une con `payments` por `order_id`.
- El ticket promedio puede ser `SUM(amount_paid) / COUNT(DISTINCT order_id)`.

## Plantilla

```sql
SELECT
    __________________ AS segment,
    COUNT(DISTINCT __________________) AS n_orders,
    SUM(__________________) AS total_pagado,
    SUM(__________________) / NULLIF(COUNT(DISTINCT __________________), 0) AS avg_ticket
FROM __________________ AS c
JOIN __________________ AS o
    ON __________________
JOIN __________________ AS p
    ON __________________
GROUP BY __________________
ORDER BY __________________ DESC;
```

## Resultado

| Segmento con mayor total pagado | Interpretación |
|---|---|
| | |

In [ ]:
-- Ejercicio 1
-- Escribe aquí tu consulta.

---
# Ejercicio 2 · Revenue por categoría (Nivel: ⭐ Básico)

## Objetivo

Calcular revenue por `category`.

Fórmula:

```text
revenue = quantity * unit_price * (1 - discount_rate)
```

## Antes de empezar

| Elemento | Respuesta |
|---|---|
| Tabla con cantidad | |
| Tabla con categoría | |
| Llave de unión | |
| Columna de agrupación | |

## Borrador

```sql
SELECT
    __________________ AS category,
    SUM(__________________ * __________________ * (1 - __________________)) AS revenue
FROM __________________ AS oi
JOIN __________________ AS pr
    ON __________________
GROUP BY __________________
ORDER BY __________________ DESC;
```

## Pregunta conceptual

¿Por qué el descuento afecta el revenue?

_____________________________________________________________________

In [ ]:
-- Ejercicio 2
-- Escribe aquí tu consulta.

---
# Ejercicio 3 · Top 5 productos por revenue (Nivel: ⭐⭐ Intermedio)

## Objetivo

Mostrar:

- `product_id`;
- `title`;
- `category`;
- revenue total.

## Control mental

- Las columnas no agregadas del `SELECT` deben aparecer en el `GROUP BY`.
- El top 5 requiere `ORDER BY revenue DESC LIMIT 5`.

## Borrador

```sql
SELECT
    __________________,
    __________________,
    __________________,
    SUM(__________________) AS revenue
FROM __________________ AS oi
JOIN __________________ AS pr
    ON __________________
GROUP BY
    __________________,
    __________________,
    __________________
ORDER BY __________________ DESC
LIMIT 5;
```

## Resultado

Producto líder: _____________________________________________________

¿El resultado es esperable? ¿Por qué?

_____________________________________________________________________

In [ ]:
-- Ejercicio 3
-- Escribe aquí tu consulta.

---
# Ejercicio 4 · Margen bruto por categoría (Nivel: ⭐⭐ Intermedio)

## Objetivo

Calcular por categoría:

- `revenue`;
- `cost`;
- `gross_profit`;
- `margin_pct`.

Fórmulas:

```text
gross_profit = revenue - cost
margin_pct = 100 * gross_profit / revenue
```

## Pregunta de control

¿Por qué conviene proteger el cálculo de margen con `NULLIF`?

_____________________________________________________________________

## Borrador

```sql
WITH categoria AS (
    SELECT
        __________________ AS category,
        SUM(__________________) AS revenue,
        SUM(__________________) AS cost
    FROM __________________ AS oi
    JOIN __________________ AS pr
        ON __________________
    GROUP BY __________________
)
SELECT
    category,
    revenue,
    cost,
    revenue - cost AS gross_profit,
    100 * (revenue - cost) / NULLIF(revenue, 0) AS margin_pct
FROM categoria
ORDER BY revenue DESC;
```

In [ ]:
-- Ejercicio 4
-- Escribe aquí tu consulta.

---
# Ejercicio 5 · Promo vs no promo (Nivel: ⭐⭐ Intermedio)

## Objetivo

Comparar órdenes con promo y sin promo.

Debes obtener:

- `grupo_promo`;
- `n_orders`;
- `total_descuento`;
- `total_pagado`;
- `avg_ticket`.

## Pista

Usa `CASE`:

```sql
CASE
    WHEN o.promo_code IS NULL THEN 'sin_promo'
    ELSE 'con_promo'
END AS grupo_promo
```

## Borrador

```sql
SELECT
    CASE
        WHEN __________________ THEN 'sin_promo'
        ELSE 'con_promo'
    END AS grupo_promo,
    COUNT(DISTINCT __________________) AS n_orders,
    SUM(__________________) AS total_descuento,
    SUM(__________________) AS total_pagado,
    SUM(__________________) / NULLIF(COUNT(DISTINCT __________________), 0) AS avg_ticket
FROM __________________ AS o
JOIN __________________ AS p
    ON __________________
GROUP BY __________________
ORDER BY __________________ DESC;
```

## Reflexión

Si las órdenes con promo venden más, eso no prueba automáticamente que la promo funciona.

Escribe una razón:

_____________________________________________________________________

In [ ]:
-- Ejercicio 5
-- Escribe aquí tu consulta.

---
# Ejercicio 6 · ROI por campaña (Nivel: ⭐⭐⭐ Avanzado)

## Objetivo

Usar la tabla `campaigns` para calcular ROI por campaña.

Debes obtener:

- `promo_code`;
- `campaign_name`;
- `spend_usd`;
- `n_orders`;
- `net_revenue`;
- `roi_pct`.

Fórmula:

```text
ROI % = 100 * (net_revenue - spend_usd) / spend_usd
```

## Borrador

```sql
SELECT
    __________________,
    __________________,
    __________________,
    COUNT(DISTINCT __________________) AS n_orders,
    SUM(__________________) AS net_revenue,
    100 * (SUM(__________________) - __________________) / NULLIF(__________________, 0) AS roi_pct
FROM __________________ AS c
LEFT JOIN __________________ AS o
    ON __________________
    AND __________________
LEFT JOIN __________________ AS pay
    ON __________________
GROUP BY
    __________________,
    __________________,
    __________________
ORDER BY __________________ DESC;
```

## Interpretación

Campaña con mejor ROI: _____________________________________________

Campaña con más volumen: ___________________________________________

¿Son la misma? ¿Qué significa?

_____________________________________________________________________

In [ ]:
-- Ejercicio 6
-- Escribe aquí tu consulta.

---
# Retos opcionales

Trabaja en:

```text
sql/08_w12_retos.sql
```

## Reto A — Estados con mayor volumen

Construye una consulta para ver estados de orden con más actividad y total pagado.

## Reto B — Top 10 clientes por gasto total

Identifica clientes de mayor valor.

## Reto C — Categorías con margen alto pero bajo volumen

Busca oportunidades de crecimiento.

## Reto D — Campañas con ROI alto pero bajo volumen

Evalúa eficiencia versus escala.

### Planeación

| Reto | Tablas | Métrica principal | Interpretación esperada |
|---|---|---|---|
| A | | | |
| B | | | |
| C | | | |
| D | | | |

In [ ]:
-- Retos opcionales
-- Escribe aquí una consulta o notas.

---
# Revisión de soluciones de referencia

Solo después de intentar los ejercicios, revisa:

```text
sql/99_soluciones_webinar12.sql
```

Úsalas para comparar:

- estructura del `JOIN`;
- columnas del `GROUP BY`;
- alias;
- regla de filtrado;
- ordenamiento;
- interpretación.

### Registro de comparación

| Ejercicio | ¿Mi lógica coincidió? | Diferencia principal |
|---|---|---|
| 0 | | |
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |
| 6 | | |

---
# Cierre

## Checklist final

- ☐ Validé el dataset antes de analizar.
- ☐ Resolví al menos 4 ejercicios.
- ☐ Usé `JOIN` correctamente.
- ☐ Usé `GROUP BY` correctamente.
- ☐ Calculé revenue y margen.
- ☐ Calculé ROI por campaña.
- ☐ Expliqué al menos un resultado en lenguaje de negocio.
- ☐ Trabajé desde archivos `.sql` en VS Code.

## Reflexión final

| Pregunta | Tu respuesta |
|---|---|
| ¿Qué ejercicio me resultó más intuitivo? | |
| ¿Cuál me costó más? | |
| ¿Qué error de SQL debo evitar? | |
| ¿Qué necesito practicar antes de la siguiente sesión? | |

## Resumen de ejercicios

| # | Ejercicio | Nivel | Estado |
|---|---|---|---|
| 0 | Estado de orden y método de pago | ⭐ | ☐ |
| 1 | Órdenes por segmento | ⭐ | ☐ |
| 2 | Revenue por categoría | ⭐ | ☐ |
| 3 | Top 5 productos | ⭐⭐ | ☐ |
| 4 | Margen bruto por categoría | ⭐⭐ | ☐ |
| 5 | Promo vs no promo | ⭐⭐ | ☐ |
| 6 | ROI por campaña | ⭐⭐⭐ | ☐ |
| A | Estados con mayor volumen | ⭐⭐⭐ | ☐ |
| B | Top 10 clientes | ⭐⭐⭐ | ☐ |
| C | Margen alto y bajo volumen | ⭐⭐⭐ | ☐ |
| D | ROI alto y bajo volumen | ⭐⭐⭐ | ☐ |

## Siguiente paso

Como más adelante usaremos Git y Jupyter notebooks, conserva esta estructura de trabajo:

```text
proyecto/
├── data/
├── sql/
├── notebooks/
└── README.md
```

Esta organización facilita versionar consultas, documentar análisis y pasar de SQL a notebooks cuando el curso lo requiera.

## Canales de apoyo

- `DATA-CO-LEARNING`: revisa los horarios de atención en la guía del estudiante.
- `DA_CONSULTA`: publica preguntas sobre contenido o proyecto usando el tag indicado por el equipo académico.
- `SPRINT FOCUS`: sesiones extra para profundizar en temas del sprint.
- `SESIONES 1:1`: agenda tutorías individuales con antelación.
- Canal de Discord de la cohorte: espacio para compartir recursos, dudas y aprendizajes.